# Example 5.1 / 1: Division, Remainders, and Nonnegative Differences

Companion notebook for **5.1 Arithmetic Operators** in the AMPL lab Documentation, section "Division, Remainders, and Nonnegative Differences". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Division, Remainders, and Nonnegative Differences

The example divides whole units into full packs, records the remainder, and prevents reported
spare capacity from becoming negative. `ampl.get_value()` evaluates a scalar expression.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param units integer >= 0;                 # Units to pack
param pack_size integer > 0;              # Units in one full pack
param capacity >= 0;                      # Space available, in units

param full_packs = units div pack_size;   # Whole packs (integer quotient)
param loose_units = units mod pack_size;  # Units left over after full packs
param spare_capacity = capacity less units;  # Unused space, never below 0

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["units"] = 29
ampl.param["pack_size"] = 12
ampl.param["capacity"] = 25

In [5]:
# run.py
for name in ["full_packs", "loose_units", "spare_capacity"]:
    print(f"{name}: {int(ampl.get_value(name))}")

full_packs: 2
loose_units: 5
spare_capacity: 0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param units := 29;
param pack_size := 12;
param capacity := 25;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display full_packs, loose_units, spare_capacity;

full_packs = 2
loose_units = 5
spare_capacity = 0

